# Brazilian E-Commerce Olist Analysis

End-to-end analysis of the Olist dataset, combining Python/Pandas analysis on the raw CSV files and SQL analysis on the Oracle database built from the same data.

## Part 1 — Data Analysis with Python and Pandas (raw CSV files)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Load the datasets

In [ ]:
def safe_read(filename):
    return pd.read_csv(filename, engine='python', on_bad_lines='warn')

orders = safe_read("olist_orders_dataset.csv")
customers = safe_read("olist_customers_dataset.csv")
order_items = safe_read("olist_order_items_dataset.csv")
payments = safe_read("olist_order_payments_dataset.csv")
reviews = safe_read("olist_order_reviews_dataset.csv")
products = safe_read("olist_products_dataset.csv")
sellers = safe_read("olist_sellers_dataset.csv")
geolocation = safe_read("olist_geolocation_dataset.csv")
category_translation = safe_read("product_category_name_translation.csv")

In [ ]:
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                  ('payments', payments), ('reviews', reviews), ('products', products),
                  ('sellers', sellers), ('geolocation', geolocation),
                  ('category_translation', category_translation)]:
    print(name, df.shape)

### Data cleaning

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [ ]:
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                  ('payments', payments), ('reviews', reviews), ('products', products),
                  ('sellers', sellers), ('geolocation', geolocation),
                  ('category_translation', category_translation)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n===== {name} =====")
        print(pd.DataFrame({"Missing Values": missing, "Percentage": (missing / len(df) * 100).round(2)}))

### Order status distribution

In [ ]:
order_status = orders["order_status"].value_counts()
order_status

In [ ]:
plt.figure(figsize=(10, 6))
order_status.plot(kind="bar", color="steelblue")
plt.title("Order Status Distribution")
plt.xlabel("Status")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

### Monthly orders trend

In [ ]:
monthly_orders = (
    orders.groupby(orders["order_purchase_timestamp"].dt.to_period("M"))
    .size()
    .reset_index(name="total_orders")
)
monthly_orders["order_purchase_timestamp"] = monthly_orders["order_purchase_timestamp"].astype(str)
monthly_orders

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(monthly_orders["order_purchase_timestamp"], monthly_orders["total_orders"], marker="o")
plt.title("Monthly Orders Trend")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Review scores

In [ ]:
average_review_score = reviews["review_score"].mean()
print("Average Review Score:", round(average_review_score, 2))

In [ ]:
review_distribution = reviews["review_score"].value_counts().sort_index()
review_distribution

In [ ]:
plt.figure(figsize=(8, 6))
plt.bar(review_distribution.index.astype(str), review_distribution.values, color="orange")
plt.title("Review Score Distribution")
plt.xlabel("Review Score")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

### Top product categories

In [ ]:
category_counts = products["product_category_name"].value_counts().head(10)
category_counts

In [ ]:
plt.figure(figsize=(12, 6))
category_counts.plot(kind="barh", color="teal")
plt.title("Top 10 Product Categories by Listing Count")
plt.xlabel("Number of Products")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
category_sales = (
    order_items
    .merge(products, on="product_id", how="left")
    .groupby("product_category_name")["price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
category_sales.columns = ["product_category", "total_sales"]
category_sales

### Payment methods

In [ ]:
payment_counts = payments["payment_type"].value_counts()
payment_counts

In [ ]:
plt.figure(figsize=(8, 6))
plt.pie(payment_counts.values, labels=payment_counts.index, autopct="%1.1f%%")
plt.title("Payment Methods Distribution")
plt.tight_layout()
plt.show()

In [ ]:
average_payment = payments["payment_value"].mean()
total_sales = payments["payment_value"].sum()
print("Average Payment Value:", round(average_payment, 2))
print("Total Sales:", round(total_sales, 2))

### Delivery time analysis

In [ ]:
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["delivery_days"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(orders["delivery_days"].dropna(), bins=30, color="steelblue", edgecolor="black")
plt.title("Distribution of Delivery Time (Days)")
plt.xlabel("Delivery Days")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

### Delivery delay vs review score

In [ ]:
orders["delivery_status"] = np.where(
    orders["order_delivered_customer_date"] > orders["order_estimated_delivery_date"],
    "Delayed", "On Time / Early"
)
delay_review = reviews.merge(orders[["order_id", "delivery_status"]], on="order_id", how="inner")
delay_review.groupby("delivery_status")["review_score"].mean()

In [ ]:
plt.figure(figsize=(8, 6))
delay_review.groupby("delivery_status")["review_score"].mean().plot(kind="bar", color=["seagreen", "crimson"])
plt.title("Average Review Score: On Time vs Delayed")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Sales by customer state

In [ ]:
customer_states = customers["customer_state"].value_counts().head(10)
customer_states

In [ ]:
plt.figure(figsize=(10, 6))
customer_states.plot(kind="bar", color="mediumpurple")
plt.title("Top 10 States by Number of Customers")
plt.xlabel("State")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

### Top sellers

In [ ]:
seller_sales = order_items.merge(payments.groupby("order_id")["payment_value"].sum().reset_index(), on="order_id", how="left")
top_sellers = seller_sales.groupby("seller_id")["payment_value"].sum().sort_values(ascending=False).head(10)
top_sellers

### Relationship validation

In [ ]:
print("===== Relationship Validation =====")
print("Orders with missing customer:", (~orders['customer_id'].isin(customers['customer_id'])).sum())
print("Order items with missing order:", (~order_items['order_id'].isin(orders['order_id'])).sum())
print("Order items with missing product:", (~order_items['product_id'].isin(products['product_id'])).sum())
print("Order items with missing seller:", (~order_items['seller_id'].isin(sellers['seller_id'])).sum())
print("Payments with missing order:", (~payments['order_id'].isin(orders['order_id'])).sum())
print("Reviews with missing order:", (~reviews['order_id'].isin(orders['order_id'])).sum())

## Part 2 — Analysis via Oracle Database

The same dataset was loaded into an Oracle database (9 tables, with primary/foreign keys and 9 analytical views). This section connects to that database and reproduces the core business analysis directly from SQL views.

In [ ]:
import oracledb

un = "PROJECT_USER"
pw = "your_password_here"
dsn = "localhost:1521/FREEPDB1"

connection = oracledb.connect(user=un, password=pw, dsn=dsn)
print("Connected to Oracle successfully")

### Monthly sales

In [ ]:
monthly_sales = pd.read_sql("SELECT * FROM VW_MONTHLY_SALES ORDER BY ORDER_MONTH", con=connection)
monthly_sales

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(monthly_sales["ORDER_MONTH"], monthly_sales["TOTAL_SALES"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Top product categories

In [ ]:
category_sales = pd.read_sql(
    "SELECT * FROM VW_TOP_CATEGORIES ORDER BY TOTAL_REVENUE DESC FETCH FIRST 10 ROWS ONLY",
    con=connection
)
category_sales

In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(category_sales["PRODUCT_CATEGORY_NAME_ENGLISH"], category_sales["TOTAL_REVENUE"], color="teal")
plt.title("Top 10 Product Categories by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Category")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Best sellers

In [ ]:
best_sellers = pd.read_sql(
    "SELECT * FROM VW_BEST_SELLERS ORDER BY TOTAL_REVENUE DESC FETCH FIRST 10 ROWS ONLY",
    con=connection
)
best_sellers

In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(best_sellers["SELLER_ID"], best_sellers["TOTAL_REVENUE"], color="darkorange")
plt.title("Top 10 Best Sellers by Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Seller ID")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Delivery time

In [ ]:
delivery_time = pd.read_sql("SELECT * FROM VW_DELIVERY_TIME", con=connection)
delivery_time["DELIVERY_DAYS"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(delivery_time["DELIVERY_DAYS"], bins=30, color="steelblue", edgecolor="black")
plt.title("Distribution of Delivery Time (Days)")
plt.xlabel("Delivery Days")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.show()

### Delivery delay vs review score

In [ ]:
delay_review = pd.read_sql(
    "SELECT DELIVERY_STATUS, AVG(REVIEW_SCORE) AS AVG_REVIEW_SCORE, COUNT(*) AS TOTAL_ORDERS "
    "FROM VW_DELAY_VS_REVIEW GROUP BY DELIVERY_STATUS",
    con=connection
)
delay_review

In [ ]:
plt.figure(figsize=(8, 6))
plt.bar(delay_review["DELIVERY_STATUS"], delay_review["AVG_REVIEW_SCORE"], color=["crimson", "seagreen"])
plt.title("Average Review Score: Delayed vs On Time")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.tight_layout()
plt.show()

### Sales by state

In [ ]:
sales_by_state = pd.read_sql(
    "SELECT * FROM VW_SALES_BY_STATE ORDER BY TOTAL_REVENUE DESC FETCH FIRST 10 ROWS ONLY",
    con=connection
)
sales_by_state

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(sales_by_state["CUSTOMER_STATE"], sales_by_state["TOTAL_REVENUE"], color="mediumpurple")
plt.title("Top 10 States by Sales Revenue")
plt.xlabel("State")
plt.ylabel("Total Revenue")
plt.tight_layout()
plt.show()

### Average order value

In [ ]:
order_value = pd.read_sql("SELECT * FROM VW_ORDER_VALUE", con=connection)
avg_order_value = round(order_value["ORDER_TOTAL"].mean(), 2)
print(f"Average Order Value: {avg_order_value}")

### Active customers

In [ ]:
active_customers = pd.read_sql("SELECT COUNT(*) AS TOTAL_ACTIVE_CUSTOMERS FROM VW_ACTIVE_CUSTOMERS", con=connection)
active_customers

### Payment methods

In [ ]:
payment_methods = pd.read_sql(
    "SELECT * FROM VW_PAYMENT_METHODS ORDER BY TOTAL_TRANSACTIONS DESC",
    con=connection
)
payment_methods

In [ ]:
plt.figure(figsize=(8, 6))
plt.pie(payment_methods["TOTAL_TRANSACTIONS"], labels=payment_methods["PAYMENT_TYPE"], autopct="%1.1f%%")
plt.title("Payment Methods Distribution")
plt.tight_layout()
plt.show()

### Close the connection

In [ ]:
connection.close()
print("Oracle connection closed")